<a href="https://colab.research.google.com/github/ZihanEvelyn/Faceplusplus-test/blob/main/Face%2B%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata

api_key = userdata.get("FACEPP_API_KEY")
api_secret = userdata.get("FACEPP_API_SECRET")

if api_key and api_secret:
    print("Success: Both Face++ credentials have been saved and can now be read by this notebook.")

Success: Both Face++ credentials have been saved and can now be read by this notebook.


In [2]:
import requests
import pandas as pd
from google.colab import userdata

In [5]:
from pathlib import Path

image_path = Path("/content/Picture 1.png")
image_name = image_path.name
image_bytes = image_path.read_bytes()

In [6]:
response = requests.post(
    "https://api-us.faceplusplus.com/facepp/v3/detect",
    data={
        "api_key": userdata.get("FACEPP_API_KEY"),
        "api_secret": userdata.get("FACEPP_API_SECRET"),
        "return_attributes": "emotion",
    },
    files={"image_file": (image_name, image_bytes)},
    timeout=60,
)

In [7]:
try:
    result = response.json()
except ValueError:
    raise RuntimeError(f"The service did not return a readable result; the HTTP status code is {response.status_code}")

In [13]:
if not response.ok or "error_message" in result:
    message = result.get("error_message", f"HTTP {response.status_code}")
    raise RuntimeError(f"Face++ Request failed ：{message}")

In [12]:
import requests
from google.colab import userdata

response = requests.post(
    "https://api-us.faceplusplus.com/facepp/v3/detect",
    data={
        "api_key": userdata.get("FACEPP_API_KEY").strip(),
        "api_secret": userdata.get("FACEPP_API_SECRET").strip(),
        "return_attributes": "emotion",
    },
    files={"image_file": (image_name, image_bytes)},
    timeout=60,
)

result = response.json()
print("本次请求的 HTTP 状态码：", response.status_code)
print("本次请求的错误信息：", result.get("error_message", "无"))

本次请求的 HTTP 状态码： 200
本次请求的错误信息： 无


In [15]:
faces = result["faces"]
print(f"Number of faces detected: {len(faces)}")

for i, face in enumerate(faces, start=1):
    scores = face.get("attributes", {}).get("emotion")
    if scores:
        print(f"\nEmotion scores for face {i} (highest to lowest):")
        for emotion, score in sorted(scores.items(), key=lambda item: -item[1]):
            print(f"{emotion}: {score:.3f}")

Number of faces detected: 1

Emotion scores for face 1 (highest to lowest):
happiness: 99.974
fear: 0.017
sadness: 0.004
anger: 0.002
disgust: 0.001
neutral: 0.001
surprise: 0.001


In [16]:
from pathlib import Path

import pandas as pd
from google.colab import files

rows = []

for face_number, face in enumerate(result["faces"], start=1):
    scores = face.get("attributes", {}).get("emotion")
    if scores:
        rows.append({
            "image_name": image_name,
            "face_number": face_number,
            **scores,
        })

if rows:
    output_file = f"{Path(image_name).stem}_facepp_emotions.csv"
    pd.DataFrame(rows).to_csv(output_file, index=False, encoding="utf-8-sig")
    files.download(output_file)
else:
    print("No facial emotion scores were returned for this image.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>